# Стадия 2. Применим ли самый простой линейный метод (оценка по числу)

**Источник данных:** `train_model.parquet` (стадия 1); первичный датасет — Kaggle S3E26, первоисточник — UCI Cirrhosis.
**Предыдущий этап:** `Stage_1_analysis.ipynb` — анализ данных и отбор 15 признаков.

**О чём эта стадия.** Самый простой способ предсказать исход — посчитать его как взвешенную сумму анализов пациента (линейная регрессия, «оценка по числу»). Метод быстр и прозрачен, поэтому его пробуют первым. Здесь мы проверяем не каче

**Почему проверка делается до обучения.** У линейного метода есть набор условий («предпосылок»), при которых он корректен. Если условие грубо нарушено, результат моделирования известен заранее, и тратить цикл обучения бессмысленно. Проверка занимает секунды и даёт тот же управленческий вывод, что полный цикл с метриками.

**Что берём в работу.** Матрицу из **15 отобранных признаков** и цели — файл `train_model.parquet`, подготовленный на стадии 1. Период наблюдения (`N_Days`) в него не входит: как показано в блоке 3.8 стадии 1, период становится известен только после наступления исхода, то есть это часть ответа, а не подсказка. Дубликаты показателей (исходные значения вместо логарифмов, возраст в днях, флаг тяжёлой стадии) тоже исключены — действует правило «одна информация — один столбец».

**Постановка проверки.** Если бы линейная регрессия применялась, цель пришлось бы кодировать числами: `C = 0` (жив), `CL = 1` (трансплантация), `D = 2` (смерть). Насколько такая кодировка соответствует реальности — вопрос первого пункта проверки.

## **1. Импорты и настройки**

In [1]:
# Сторонние библиотеки
import os

import numpy as np
import pandas as pd

pd.set_option('display.width', 200)

## **2. Загрузка данных**

Данные берутся из `data/processed/train_model.parquet` — это итоговый набор стадии 1 (блок 3.12): 15 признаков модели плюс целевая переменная `Status`, 7 905 строк после очистки. Повторная очистка и отбор не требуются — решения приняты и обоснованы на стадии 1.

In [2]:
# Корень проекта: папка с data/, найденная вверх по дереву от текущей директории
def find_project_root(start=None):
    d = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(d, 'data')):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            raise FileNotFoundError('Папка data/ не найдена вверх по дереву от ' + d)
        d = parent


PROJECT_ROOT = find_project_root()
df = pd.read_parquet(os.path.join(PROJECT_ROOT, 'data', 'processed', 'train_model.parquet'))
print(f'Строк: {len(df)}, столбцов: {df.shape[1]} (15 признаков + цель)')
df.head(3)

Строк: 7905, столбцов: 16 (15 признаков + цель)


,log_Bilirubin,log_Copper,log_Alk_Phos,log_SGOT,log_Cholesterol,log_Tryglicerides,Albumin,Platelets,Prothrombin,Age_years,Stage,Edema_num,Sex_M,Hepatomegaly_Y,Spiders_Y,Status
0,1.193922,5.153292,7.379008,5.197391,5.758902,4.158883,3.35,394.0,9.7,59.0,3.0,0.0,1,0,0,D
1,0.641854,4.158883,7.273093,4.911551,5.899897,4.488636,3.54,361.0,11.0,52.7,3.0,0.0,0,0,0,C
2,1.458615,4.882802,6.937314,4.790404,5.703782,3.931826,3.55,199.0,11.7,37.6,4.0,1.0,0,1,1,D


### Целевая переменная

In [3]:
# Кодируем Status в порядковую числовую шкалу: C=0, CL=1, D=2
status_map = {'C': 0, 'CL': 1, 'D': 2}
y = df['Status'].astype(str).map(status_map)

print('Распределение классов цели:')
display(y.map({0: 'C (жив)', 1: 'CL (трансплантация)', 2: 'D (умер)'})
        .value_counts(normalize=True).sort_index().rename('доля').round(4).to_frame())

Распределение классов цели:


,доля
Status,
C (жив),0.6281
CL (трансплантация),0.0348
D (умер),0.3371


Матрица признаков — это все столбцы файла, кроме цели. Никакого дополнительного кодирования не нужно: бинарные признаки (мужской пол, увеличенная печень, сосудистые «паучки») уже приведены к 0/1 на стадии 1, порядковая шкала отёков — к числам 0 / 0.5 / 1, скошенные лабораторные показатели — к логарифмам.

Напоминание, почему в наборе нет остальных столбцов (обоснование — блоки 2.5, 3.8, 3.10, 3.11 стадии 1):

| Причина исключения | Столбцы |
|---|---|
| Дублируют выбранный вариант одного и того же факта | `Bilirubin`, `Copper`, `Alk_Phos`, `SGOT`, `Cholesterol`, `Tryglicerides`, `Age`, `Edema`, `is_severe`, `Ascites` |
| Известен только после наступления исхода | `N_Days` (период наблюдения) |
| Не связан с исходом / технический | `Drug`, `id` |
| Прямая утечка цели | `is_dead` |

In [4]:
X = df.drop(columns=['Status']).astype(float)

print(f'Матрица признаков: {X.shape[0]} строк x {X.shape[1]} столбцов')
print('Столбцы:', list(X.columns))

Матрица признаков: 7905 строк x 15 столбцов
Столбцы: ['log_Bilirubin', 'log_Copper', 'log_Alk_Phos', 'log_SGOT', 'log_Cholesterol', 'log_Tryglicerides', 'Albumin', 'Platelets', 'Prothrombin', 'Age_years', 'Stage', 'Edema_num', 'Sex_M', 'Hepatomegaly_Y', 'Spiders_Y']


## **4. Быстрая оценка применимости линейной регрессии**

У линейной регрессии есть набор условий, которые можно проверить за секунды — **без** обучения модели, разбиения на части и анализа остатков. Если условие грубо нарушено, результат моделирования предсказуем заранее. Проверяем четыре вещи на всей выборке (модели нет, поэтому подглядывания в ответ не грозит):

1. **Тип и распределение цели** — линейная регрессия рассчитана на непрерывную цель на всей числовой прямой.
2. **Наличие линейной связи** «признак ↔ цель» (корреляция Пирсона).
3. **Линейность этой связи** — сравнение Пирсона и Спирмена.
4. **Мультиколлинеарность** между признаками — близкие пары делают веса нестабильными.

> **Пирсон против Спирмена.** Пирсон меряет силу *линейной* связи, Спирмен — силу *монотонной* (любой возрастающей/убывающей). Если Спирмен заметно больше Пирсона, связь есть, но она нелинейна — линейная регрессия её недоиспользует.

In [5]:
# Проверка 1. Тип и распределение цели
n_classes = y.nunique()
print(f'Целевая переменная принимает {n_classes} значения: {dict(y.value_counts().sort_index())}')

if n_classes <= 10:
    print('=> Цель ДИСКРЕТНАЯ (мало значений). Линейная регрессия формально')
    print('   не предназначена для такой цели: она вернёт значения вне {0, 1, 2},')
    print('   а остатки не будут нормальными. Это ограничение, а не ошибка данных.')
else:
    print('=> Цель почти непрерывная — предпосылка линейной регрессии выполнена.')

Целевая переменная принимает 3 значения: {0: np.int64(4965), 1: np.int64(275), 2: np.int64(2665)}
=> Цель ДИСКРЕТНАЯ (мало значений). Линейная регрессия формально
   не предназначена для такой цели: она вернёт значения вне {0, 1, 2},
   а остатки не будут нормальными. Это ограничение, а не ошибка данных.


In [6]:
# Проверка 2. Линейная связь каждого признака с целью (Пирсон)
corr_with_target = X.corrwith(y).sort_values(key=abs, ascending=False)
print('Топ-8 линейных связей с целью:')
display(corr_with_target.head(8).round(3).to_frame('r (Пирсон)'))

max_abs = corr_with_target.abs().max()
if max_abs < 0.2:
    verdict2 = 'слабые: почти все |r| < 0.2 — линейной модели брать практически нечего'
elif max_abs < 0.4:
    verdict2 = 'умеренные: |r| ~ 0.2-0.4 — линейная связь есть, но сигнал не сильный'
else:
    verdict2 = 'сильные: есть признаки с |r| > 0.4 — линейная модель вытянет заметную часть'
print(f'\n=> Связи с целью: {verdict2}. Максимум |r| = {max_abs:.2f} ({corr_with_target.abs().idxmax()}).')

Топ-8 линейных связей с целью:


,r (Пирсон)
log_Bilirubin,0.534
log_Copper,0.414
Hepatomegaly_Y,0.396
Prothrombin,0.388
Stage,0.347
Edema_num,0.337
log_SGOT,0.328
Spiders_Y,0.324



=> Связи с целью: сильные: есть признаки с |r| > 0.4 — линейная модель вытянет заметную часть. Максимум |r| = 0.53 (log_Bilirubin).


In [7]:
# Проверка 3. Линейность связи: Пирсон против Спирмена для сильнейших признаков
top_feats = corr_with_target.head(6).index
lin_check = pd.DataFrame({
    'Пирсон': [X[f].corr(y) for f in top_feats],
    'Спирмен': [X[f].corr(y, method='spearman') for f in top_feats],
}, index=top_feats)
lin_check['разрыв'] = (lin_check['Спирмен'] - lin_check['Пирсон']).abs()
display(lin_check.round(3))

print('=> Разрыв у всех сильнейших признаков не больше 0.024: связи почти линейны.')
print('   Это ожидаемо: скошенные показатели уже приведены к логарифмической шкале')
print('   на стадии 1 (блок 2.5), и именно log-варианты лежат в матрице признаков.')
print('   Для сравнения: неубранный исходный билирубин давал разрыв 0.11 —')
print('   проверка 3 на исходных данных выявила бы нелинейность.')

,Пирсон,Спирмен,разрыв
log_Bilirubin,0.534,0.540,0.006
log_Copper,0.414,0.438,0.024
Hepatomegaly_Y,0.396,0.397,0.000
Prothrombin,0.388,0.382,0.006
Stage,0.347,0.363,0.016
Edema_num,0.337,0.327,0.009


=> Разрыв у всех сильнейших признаков не больше 0.024: связи почти линейны.
   Это ожидаемо: скошенные показатели уже приведены к логарифмической шкале
   на стадии 1 (блок 2.5), и именно log-варианты лежат в матрице признаков.
   Для сравнения: неубранный исходный билирубин давал разрыв 0.11 —
   проверка 3 на исходных данных выявила бы нелинейность.


In [8]:
# Проверка 4. Мультиколлинеарность между признаками
feat_corr = X.corr()
mask = np.triu(np.ones_like(feat_corr, dtype=bool), k=1)
pairs = feat_corr.where(mask).stack().sort_values(key=abs, ascending=False)

print('Топ-6 пар признаков по |r|:')
display(pairs.head(6).round(3).to_frame('|r|'))

n_08 = int((pairs.abs() > 0.8).sum())
n_06 = int((pairs.abs() > 0.6).sum())
print(f'\n=> Пар с |r| > 0.8: {n_08}. Пар с |r| > 0.6: {n_06}.')
print(f'Сильнейшая пара: «{pairs.index[0][0]} ↔ {pairs.index[0][1]}» |r| = {pairs.iloc[0]:.3f}')
print('Это клинически осмысленное сочетание (билирубин и медь растут при поражении')
print('печени), а не дублирование. Предпосылка выполнена: проблема начинается от ~0.8.')
print('\nНапоминание: дубликаты (показатель и его логарифм, асцит и отёки) удалены')
print('на стадии 1 — поэтому здесь видны только настоящие связи данных.')

Топ-6 пар признаков по |r|:


,,|r|
log_Bilirubin,log_Copper,0.538
Stage,Hepatomegaly_Y,0.498
log_Bilirubin,log_SGOT,0.464
log_Copper,log_SGOT,0.403
log_Bilirubin,Hepatomegaly_Y,0.387
log_SGOT,log_Cholesterol,0.352



=> Пар с |r| > 0.8: 0. Пар с |r| > 0.6: 0.
Сильнейшая пара: «log_Bilirubin ↔ log_Copper» |r| = 0.538
Это клинически осмысленное сочетание (билирубин и медь растут при поражении
печени), а не дублирование. Предпосылка выполнена: проблема начинается от ~0.8.

Напоминание: дубликаты (показатель и его логарифм, асцит и отёки) удалены
на стадии 1 — поэтому здесь видны только настоящие связи данных.


### Вердикт по предпосылкам (до всякого обучения модели)

По четырём быстрым проверкам заранее видно:

| Предпосылка | Что показали метрики | Вывод для линейной регрессии |
|---|---|---|
| Непрерывность цели | 3 значения (0/1/2) | **Грубо нарушена**: предсказания выйдут за диапазон классов, остатки не будут нормальными |
| Линейная связь с целью | максимум \|r\| ≈ 0.53 (`log_Bilirubin`) | Сигнал есть — признаки информатны, но связность умеренная |
| Линейность связи | разрыв Пирсона и Спирмена ≤ 0.024 у сильнейших признаков | Предпосылка выполнена — спасибо лог-преобразованиям и отбору стадии 1 |
| Мультиколлинеарность | ни одной пары \|r\| > 0.6, сильнейшая `log_Bilirubin ↔ log_Copper` = 0.538 | Предпосылка выполнена |

Три предпосылки из четырёх соблюдены — и это хороший знак для самих данных. Но главный пункт — первый, и он неустраним: **дискретная категориальная цель не является непрерывной величиной**. Никакая работа с признаками (логи, отбор, масштабирование) это не лечит, потому что нарушение — в самой постановке задачи, а не в данных.

> **Вывод по методике:** линейную регрессию для задач классификации не используют. Корректный инструмент для предсказания `Status` — многоклассовая **логистическая регрессия** (и далее ансамбли), с метриками классификации. Быстрая проверка заняла секунды и дала тот же управленческий вывод, что занял бы полный цикл моделирования.

## **5. Выводы**

### Ответ на вопрос задачи

**Можно ли применить линейную регрессию (оценку по числу) для предсказания клинического статуса?** Проверка предпосылок даёт отрицательный ответ, и он следует из структуры задачи, а не из качества данных:

1. **Цель не является непрерывной.** `Status` принимает три значения, разделённые произвольным числовым шагом. Линейная регрессия по природе выдаёт любое вещественное число: предсказания вроде −0.4 или 2.7 не соответствуют ни одному классу, а промежуточные значения (1.3) не имеют клинического смысла — «полутрансплантации» не существует.

2. **Кодировка числами вводит ложную меру расстояния.** Модель была бы вынуждена считать, что расстояние «жив → трансплантация» равно расстоянию «трансплантация → смерть». Трансплантация (`CL`, всего 3.5 % наблюдений) — не «промежуточная тяжесть» между выживанием и смертью, а отдельный исход, зависящий в том числе от решения врача и доступности донорского органа.

3. **Метрики качества регрессии были бы неинтерпретируемы.** Доля объяснённого разброса (R²), средняя абсолютная ошибка (MAE) и среднеквадратичная ошибка (RMSE) измеряют ошибку в единицах условной шкалы 0–2; «ошибка 0.5» не отвечает на практический вопрос «выживёт ли пациент». Для классификации нужны доля верных ответов, логарифмическая потеря и кривая качества (ROC AUC).

4. **Сами данные для моделирования пригодны.** Признаки информативны (максимум связи с целью |r| = 0.53 у `log_Bilirubin`), связи почти линейны (разрыв Пирсона и Спирмена ≤ 0.024), близких пар между признаками нет (сильнейшая — `log_Bilirubin ↔ log_Copper` = 0.538, клинически осмысленное сочетание маркёров поражения печени). Вывод «нет» относится к выбору метода, а не к датасету.

5. **Практический итог метода:** отказ от линейной регрессии здесь принят за секунды корреляционных расчётов, без единого обучения модели. Проверка предпосылок сэкономила цикл моделирования и защитила от некорректной интерпретации его результатов.

### Глоссарий терминов

| Термин | Что означает | Зачем нужен |
|--------|-------------|-------------|
| **Целевая переменная** | То, что модель пытается предсказать (здесь — `Status`) | Без неё модель не знает, чему учиться |
| **Признаки** | Входы модели: лабораторные показатели, стадия, демография (15 столбцов) | Информация, на основе которой строится предсказание |
| **Категориальная цель** | Цель задаётся конечным набором классов без числового расстояния между ними | Определяет выбор класса моделей: нужны классификаторы, а не регрессии |
| **Порядковая кодировка** | Преобразование `C→0, CL→1, D→2`: числа упорядочены, но «шаг» между ними произволен | Единственный способ подать категорию в линейную регрессию — и источник её некорректности здесь |
| **Предпосылка модели** | Свойство данных, при котором метод корректен (непрерывность цели, линейность связи, отсутствие близких признаков) | Проверка предпосылок отвечает на вопрос «можно ли применять метод» до вопроса «насколько качественно» |
| **Корреляция Пирсона** | Сила *линейной* связи, от −1 до 1 | Показывает, есть ли что «вытягивать» линейной модели |
| **Корреляция Спирмена** | Сила *монотонной* связи (по порядку значений) | Сравнение с Пирсоном выявляет нелинейность: большой разрыв = связь есть, но не линейная |
| **Мультиколлинеарность** | Высокая корреляция между признаками | Делает веса модели нестабильными: знак может «переворачиваться» |
| **Доля объяснённого разброса (R²)** | Какая часть колебаний цели описывается моделью, от 0 до 1 | Мера качества регрессии; для категорий не применима |
| **Логарифмическая потеря (log-loss)** | Штраф за уверенность в неверном ответе классификатора | Главная метрика качества вероятностного прогноза |
| **Утечка цели** | В признаки попадает информация, прямо вычисляемая из цели (`is_dead` — это `Status == 'D'`) | Модель «предсказывает» известное, метрики становятся фиктивными |
| **Период наблюдения (`N_Days`)** | Сколько дней пациент состоял под наблюдением до развязки | Известен только после исхода — на стадии 1 исключён из признаков как следствие, а не причина |